In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

### Model calling without Schema

In [24]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=1.0,
    timeout=200,
    max_retries=2,
)


### Langchain support 4 types of structured output
- TypedDict
- Dataclass
- Pydantic Model
- JSON Schema

Use method `with_structured_output` to get output is some structured format

#### TypedDict Structured Output
Read about typed dict https://typing.python.org/en/latest/spec/typeddict.html
Ideal when you don’t need runtime validation.

In [25]:
from typing_extensions import TypedDict,Annotated
from langchain_core.prompts import PromptTemplate
class FactsModel(TypedDict):
    facts: Annotated[list[str],..., "A list of 5 facts about the given word."]
    word: Annotated[str, ...,"The word for which the facts are being provided."]

structured_output = model.with_structured_output(FactsModel)

prompt = PromptTemplate.from_template(template="Give me 5 facts about {word}.")
prompt= prompt.format(word="Tenali Raman")

result = structured_output.invoke(prompt)


In [26]:
print(result.get('facts'))
print(result.get('word'))

['Tenali Raman, born Garlapati Ramakrishna, was a famous 16th-century Telugu poet and court jester in the Vijayanagara Empire.', 'He served in the court of Emperor Krishnadevaraya and was celebrated as one of the Ashtadiggajas, the eight legendary scholars of the court.', 'He hailed from the town of Tenali in Andhra Pradesh, which is how he acquired the moniker Tenali.', 'Renowned for his extraordinary wit, humor, and sharp intellect, his clever problem-solving stories are highly popular in Indian folklore.', 'He was a devout follower of Vaishnavism, and his major literary work, Panduranga Mahatmyam, is considered one of the Pancha Kavyas of Telugu literature.']
Tenali Raman


#### Dataclass Structured Output
Please read this if you dont know what data classes are in python
https://www.datacamp.com/tutorial/python-data-classes

The @dataclass decorator in Python is a convenient way to create classes with automatically generated boilerplate Dunder methods like constructors (__init__), equality checks (__eq__), and string representations (__repr__).

*Official docs* https://docs.python.org/3/library/dataclasses.html#module-contents

#### Google gemini dont support Dataclass

## Pydantic Model Structured Output
Pydantic give wings to your model classes.

*Please follow pydantic tutorial*
- https://medium.com/@marcnealer/a-practical-guide-to-using-pydantic-8aafa7feebf6
- https://pydantic.dev/docs/validation/latest/concepts/models/

In [34]:
from langchain_core.prompts import PromptTemplate
from typing import List
from pydantic import BaseModel, Field, json_schema


class GeminiStructuredResponse(BaseModel):
    facts:List[str] = Field(..., description="A list of 5 facts about the given word.")
    word: str = Field(..., description="The word for which the facts are being provided.")

structured_output = model.with_structured_output(GeminiStructuredResponse)

prompt = PromptTemplate.from_template(template="Give me 5 facts about {word}.")
prompt= prompt.format(word="Tenali Raman")

result = structured_output.invoke(prompt)

print(f"Facts : {result.facts}")
print(f"Fact word : {result.word}")

Facts : ["Tenali Raman was a Telugu poet and a prominent member of the 'Ashtadiggajas', the group of eight poets in the court of King Krishnadevaraya of the Vijayanagara Empire.", 'He was born in a village named Tenali in Andhra Pradesh, which became part of his name.', "Apart from his wit and humor, he was a scholar of high standing and wrote 'Panduranga Mahatmyam', a major work of Telugu literature.", 'Popular folklore suggests he received his legendary intellect and humor as a boon from the Goddess Kali after making her laugh.', "He is often compared to Birbal of Akbar's court due to his similar role as a witty advisor who solved complex problems using humor and intelligence."]
Fact word : Tenali Raman


#### JSON structured output
Below is the example schema
```
json_schema = {
    "type": "object",
    "description": "Fact information JSON schema",
    "properties": {
        "word": {"type": "string", "description": "The word for which the facts are being provided."},
        "facts": {
            "type": "array",
            "description": "List of facts returned by the LLM.",
            "items": {
                "type": "string"
            }
        }
    },
    "required": ["word", "facts"]
}
```

In [38]:
json_schema = {
    "type": "object",
    "description": "Fact information JSON schema",
    "properties": {
        "word": {"type": "string", "description": "The word for which the facts are being provided."},
        "facts": {
            "type": "array",
            "description": "List of facts returned by the LLM.",
            "items": {
                "type": "string"
            }
        }
    },
    "required": ["word", "facts"]
}
structured_output = model.with_structured_output(json_schema)

prompt = PromptTemplate.from_template(template="Give me 5 facts about {word}.")
# prompt= prompt.format(word="Tenali Raman")

chain = prompt | structured_output
result = chain.invoke({"word":"Tenali Raman"})

print(result)

{'word': 'Tenali Raman', 'facts': ['Tenali Raman was a legendary Telugu poet and scholar who served as an advisor and court jester to King Krishnadevaraya of the Vijayanagara Empire.', 'He was one of the Ashtadiggajas, the group of eight exceptional Telugu poets in the royal court.', "He was famously given the title of 'Vikatakavi', a palindrome in Telugu that translates to a jesting or clownish poet.", 'Originally named Garlapati Ramakrishna, he came to be known as Tenali Raman after his ancestral village of Tenali in Andhra Pradesh.', "Beyond his witty stories, he was an accomplished writer whose masterpiece 'Panduranga Mahatmyam' is considered one of the five great classics of Telugu literature."]}
